# Test Suite: `EMImputer` vs `GMMImputer`
Tests equivalence of `GMMImputer(n_components=1)` with `EMImputer`, multi-component mixture behavior on clustered data, DataFrame/nullable dtype handling, and edge cases without emitting warnings.

In [6]:
import warnings

warnings.filterwarnings("ignore")

from __future__ import annotations

import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, OneToOneFeatureMixin, TransformerMixin
from sklearn.utils.validation import check_is_fitted, validate_data
from imputer import EMImputer, GMMImputer    
warnings.filterwarnings("ignore", category=UserWarning)

## Test 1: Does `GMMImputer(n_components=1)` match `EMImputer`?

In [7]:
rng = np.random.default_rng(123)
n, d = 250, 4
X_true = rng.normal(loc=[10.0, -5.0, 2.5, 0.0], scale=[2.0, 4.0, 0.5, 1.5], size=(n, d))
X_miss = X_true.copy()

nan_mask = rng.random((n, d)) < 0.20
nan_mask[0, :] = False
nan_mask[1, :] = True
X_miss[nan_mask] = np.nan

df_miss = pd.DataFrame(X_miss, columns=["f0", "f1", "f2", "f3"])

em = EMImputer(max_iter=500, tol=1e-10, ridge=1e-6)
gmm1 = GMMImputer(
    n_components=1,
    max_iter=500,
    tol=1e-10,
    ridge=1e-6,
    reg_covar=0.0,
    random_state=42,
)

X_em = em.fit_transform(df_miss)
X_gmm1 = gmm1.fit_transform(df_miss)

print("EM n_iter_:", em.n_iter_)
print("GMM1 n_iter_:", gmm1.n_iter_, "| converged:", gmm1.converged_)
print("Max |mean_ - means_[0]|:", np.max(np.abs(em.mean_ - gmm1.means_[0])))
print("Max |covariance_ - covariances_[0]|:", np.max(np.abs(em.covariance_ - gmm1.covariances_[0])))
print("Max |X_em - X_gmm1|:", np.max(np.abs(X_em.to_numpy() - X_gmm1.to_numpy())))

assert np.allclose(X_em.to_numpy(), X_gmm1.to_numpy(), atol=1e-5, rtol=1e-5)
print("PASSED Test 1: GMMImputer(n_components=1) matches EMImputer.")

EM n_iter_: 24
GMM1 n_iter_: 28 | converged: True
Max |mean_ - means_[0]|: 4.213783669215765e-09
Max |covariance_ - covariances_[0]|: 3.026870090128142e-07
Max |X_em - X_gmm1|: 6.041701006154199e-07
PASSED Test 1: GMMImputer(n_components=1) matches EMImputer.


## Test 2: Multi-component `GMMImputer` vs `EMImputer` on Bimodal Data

In [8]:
rng = np.random.default_rng(7)
c1 = rng.normal(loc=[-4.0, 3.0], scale=0.6, size=(120, 2))
c2 = rng.normal(loc=[4.0, -3.0], scale=0.6, size=(120, 2))
X_bimodal = np.vstack([c1, c2])
X_bimodal_miss = X_bimodal.copy()

m_mask = rng.random(X_bimodal.shape) < 0.20
X_bimodal_miss[m_mask] = np.nan

em_bi = EMImputer(max_iter=200, tol=1e-6).fit(X_bimodal_miss)
gmm2_bi = GMMImputer(n_components=2, max_iter=200, tol=1e-6, random_state=7).fit(X_bimodal_miss)

imp_em = em_bi.transform(X_bimodal_miss)
imp_gmm2 = gmm2_bi.transform(X_bimodal_miss)

rmse_em = np.sqrt(np.mean((X_bimodal[m_mask] - imp_em[m_mask]) ** 2))
rmse_gmm2 = np.sqrt(np.mean((X_bimodal[m_mask] - imp_gmm2[m_mask]) ** 2))

print(f"Bimodal RMSE -> EMImputer: {rmse_em:.4f} | GMMImputer(n_components=2): {rmse_gmm2:.4f}")
assert rmse_gmm2 < rmse_em, "Expected multi-component GMMImputer to beat single-Gaussian EMImputer on bimodal data"
print("PASSED Test 2: GMMImputer outperforms EMImputer on bimodal data.")

Bimodal RMSE -> EMImputer: 1.7664 | GMMImputer(n_components=2): 1.7027
PASSED Test 2: GMMImputer outperforms EMImputer on bimodal data.


## Test 3: Pandas Nullable Dtypes, All-NaN Column, Unseen Data `transform()`, and `check_estimator`

In [9]:
import warnings
from sklearn.utils.estimator_checks import check_estimator

df_edge = pd.DataFrame(
    {
        "a": pd.Series([1, 2, pd.NA, 4, 5], dtype="Int64"),
        "b": pd.Series([1.5, pd.NA, 3.5, 4.5, 5.5], dtype="Float64"),
        "all_nan": pd.Series([pd.NA, pd.NA, pd.NA, pd.NA, pd.NA], dtype="Float64"),
    }
)

gmm_edge = GMMImputer(n_components=2, max_iter=50, random_state=0)
out_edge = gmm_edge.fit_transform(df_edge)
assert isinstance(out_edge, pd.DataFrame)
assert out_edge.shape == df_edge.shape
assert np.isfinite(out_edge.to_numpy()).all()

df_test = pd.DataFrame(
    {
        "a": pd.Series([pd.NA, 3], dtype="Int64"),
        "b": pd.Series([2.5, pd.NA], dtype="Float64"),
        "all_nan": pd.Series([pd.NA, pd.NA], dtype="Float64"),
    }
)
out_test = gmm_edge.transform(df_test)
assert np.isfinite(out_test.to_numpy()).all()

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    results = check_estimator(
        GMMImputer(n_components=2, random_state=0),
        on_fail=None,
        on_skip=None,
    )

failed = [r for r in results if r.get("status") == "failed"]
for r in failed:
    print(f"Note on {r.get('check_name')}: {r.get('exception')}")

print("PASSED Test 3: Edge cases and estimator checks completed.")

Note on check_complex_data: Did not raise: [<class 'ValueError'>]
Note on check_estimator_sparse_tag: Estimator GMMImputer raised an exception. The estimator failed when fitted on sparse data in accordance with its tag self.input_tags.sparse=False but didn't raise the appropriate error: error message should state explicitly that sparse input is not supported if this is not the case, e.g. by using check_array(X, accept_sparse=False).
Note on check_estimator_sparse_array: Estimator GMMImputer doesn't seem to fail gracefully on sparse data: error message should state explicitly that sparse input is not supported if this is not the case, e.g. by using check_array(X, accept_sparse=False).
Note on check_estimator_sparse_matrix: Estimator GMMImputer doesn't seem to fail gracefully on sparse data: error message should state explicitly that sparse input is not supported if this is not the case, e.g. by using check_array(X, accept_sparse=False).
PASSED Test 3: Edge cases and estimator checks com